# 01 · Data collection (Alpaca → bucket table)

**Goal:** turn raw trades/quotes into one row per *(ticker, day, slot)* bucket, without ever storing tick data.

For each (ticker, day) we pull trades + quotes, match every trade to its prevailing quote, compute
**effective spread** `2·|P − M| / M` (bps) and the feature candidates for the three families in the proposal
(spread/liquidity, price deviation, oscillation), then cache ≤7 rows. Downloads are **resumable**.

Buckets: 9:30–16:00 ET → six 1-hour slots + one 30-minute slot. With the weekday this is the market analogue of
the paper's *hour-of-week* bucket.

**Needs:** `ALPACA_API_KEY`, `ALPACA_SECRET_KEY` environment variables (see README).

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from htt import config as C

import os, logging
from htt import data
logging.basicConfig(level=logging.INFO)
assert os.getenv("ALPACA_API_KEY") and os.getenv("ALPACA_SECRET_KEY"), "Set Alpaca keys (see README)"

## Configuration
Start with `SMOKE_TEST = True` to verify access and data sufficiency (a proposal to-do), then flip it off.

In [ ]:
SMOKE_TEST = True
INCLUDE_CRYPTO = False      # stretch goal for RQ4; crypto quotes are very large, so try a few days first

days = [d.strftime("%Y-%m-%d") for d in pd.bdate_range(C.START, C.END)]
if SMOKE_TEST:
    universe = {"tech": ["AAPL", "MSFT"], "fin": ["JPM", "BAC"]}
    days = days[:10]
else:
    universe = {k: C.SECTORS[k]["symbols"] for k in ("tech", "fin")}
benchmarks = ["XLK", "XLF"]
print(len(days), "business days;", sum(map(len, universe.values())), "stocks;", "feed =", C.FEED)

## Sanity check: one ticker-day, end to end

In [ ]:
client = data.get_client("stock")
tr, qu = data.fetch_day(client, "AAPL", days[1], "stock")
print(len(tr), "trades,", len(qu), "quotes")
summ = data.summarize_day(tr, qu, "AAPL", "stock")
summ[["slot", "n_trades", "n_matched", "eff_vw", "eff_med", "qspread_mean", "depth_mean", "range_bps"]]

If `n_matched` is far below `n_trades`, many trades have no quote within `MAX_QUOTE_AGE_S`; raise that
threshold in `config.py` or consider a different feed. Check the **data-sufficiency** question here before the full pull.

In [ ]:
failed = []
for sector, syms in universe.items():
    failed += data.collect(syms, days, kind="stock")
failed += data.collect(benchmarks, days, kind="stock")
print("failed symbol-days (re-run this cell to retry):", failed[:10], "...")

In [ ]:
if INCLUDE_CRYPTO:
    cdays = [d.strftime("%Y-%m-%d") for d in pd.date_range(C.START, C.END)]
    cdays = cdays[:7] if SMOKE_TEST else cdays
    failed_c = data.collect(C.CRYPTO + ["BTC/USD"], cdays, kind="crypto")
    print("crypto failures:", failed_c[:10])

## Assemble + quality checks

In [ ]:
stocks = data.load_cached(sum(universe.values(), []) + benchmarks, "stock")
frames = [stocks]
if INCLUDE_CRYPTO:
    frames.append(data.load_cached(C.CRYPTO + ["BTC/USD"], "crypto"))
raw = pd.concat(frames, ignore_index=True)
raw.to_parquet(C.DATA_PROC / "raw_buckets.parquet")
print(raw.shape, raw.symbol.nunique(), "symbols,", raw.date.nunique(), "days")

qc = (raw.groupby("symbol").agg(days=("date", "nunique"), buckets=("slot", "size"),
                                median_trades=("n_trades", "median"), median_eff_bps=("eff_vw", "median")))
qc

### Does the known U-shape (McInish & Wood) show up?
Median effective spread by slot, each ticker divided by its own median.

In [ ]:
s = raw[~raw.symbol.isin(benchmarks + ["BTC/USD"]) & raw.symbol.isin(sum(universe.values(), []))].copy()
s["rel"] = s.eff_vw / s.groupby("symbol").eff_vw.transform("median")
s["weekday"] = s.date.dt.weekday
heat = s.pivot_table(index="weekday", columns="slot", values="rel", aggfunc="median")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
s.groupby("slot").rel.median().plot(marker="o", ax=ax[0], title="Relative effective spread by slot")
im = ax[1].imshow(heat, aspect="auto", cmap="viridis"); ax[1].set_title("weekday x slot")
ax[1].set_xticks(range(heat.shape[1]), heat.columns); ax[1].set_yticks(range(heat.shape[0]), ["Mon","Tue","Wed","Thu","Fri"][:heat.shape[0]])
plt.colorbar(im, ax=ax[1]); plt.tight_layout()